In [1]:
import pandas as pd

In [2]:
def calculate_metrics(rows):
    # Convert list of dicts to DataFrame
    df = pd.DataFrame(rows)
    
    # Pre-calculate totals for each row
    df["Support"] = df["TP"] + df["FN"]
    df["Pred_Count"] = df["TP"] + df["FP"]

    # Calculate individual metrics
    # We use .where() or replacement to handle division by zero safely
    df["Precision"] = df["TP"] / df["Pred_Count"].replace(0, 1)
    df["Recall"] = df["TP"] / df["Support"].replace(0, 1)
    
    denom = (df["Precision"] + df["Recall"]).replace(0, 1)
    df["F1-Score"] = 2 * (df["Precision"] * df["Recall"]) / denom

    # Calculate Weighted Averages based on Support
    total_support = df["Support"].sum()
    
    if total_support > 0:
        w_prec = (df["Precision"] * df["Support"]).sum() / total_support
        w_rec  = (df["Recall"] * df["Support"]).sum() / total_support
        w_f1   = (df["F1-Score"] * df["Support"]).sum() / total_support
    else:
        w_prec = w_rec = w_f1 = 0.0

    # Create the summary row
    avg_row = pd.DataFrame([{
        "Antipattern": "WEIGHTED AVERAGE",
        "TP": df["TP"].sum(),
        "FP": df["FP"].sum(),
        "FN": df["FN"].sum(),
        "Support": total_support,
        "Pred_Count": df["Pred_Count"].sum(),
        "Precision": w_prec,
        "Recall": w_rec,
        "F1-Score": w_f1
    }])
    
    # Combine and return
    return pd.concat([df, avg_row], ignore_index=True).round(4)

In [16]:
data = [
    {"Antipattern": "Implicit Columns", "TP": 256, "FP": 1, "FN": 15},
    {"Antipattern": "ID Required", "TP": 95, "FP": 5, "FN": 7},
    {"Antipattern": "Keyless Entry", "TP": 40, "FP": 25, "FN": 0},
    {"Antipattern": "Fear of the Unknown", "TP": 42, "FP": 1, "FN": 17},
    {"Antipattern": "31 Flavors", "TP": 38, "FP": 1, "FN": 1},
    {"Antipattern": "Poor Man's Search Engine", "TP": 13, "FP": 4, "FN": 8},
    {"Antipattern": "Rounding Errors", "TP": 13, "FP": 0, "FN": 3}
]
calculate_metrics(data)

,Antipattern,TP,FP,FN,Support,Pred_Count,Precision,Recall,F1-Score
0,Implicit Columns,256,1,15,271,257,0.9961,0.9446,0.9697
1,ID Required,95,5,7,102,100,0.9500,0.9314,0.9406
2,Keyless Entry,40,25,0,40,65,0.6154,1.0000,0.7619
3,Fear of the Unknown,42,1,17,59,43,0.9767,0.7119,0.8235
4,31 Flavors,38,1,1,39,39,0.9744,0.9744,0.9744
5,Poor Man's Search Engine,13,4,8,21,17,0.7647,0.6190,0.6842
6,Rounding Errors,13,0,3,16,13,1.0000,0.8125,0.8966
7,WEIGHTED AVERAGE,497,37,51,548,534,0.9473,0.9069,0.9206
